# Dense captioning generation

1. Get image url from roboflow
2. Query openrouter with image and get dense captinos
3. store url and dense captions in json format

In [23]:
import roboflow_client as rc
import dotenv
import os
import json
import time
import tqdm
import base64
from io import BytesIO
from PIL import Image
import requests
from tqdm.notebook import tqdm as tqdmnote
import openai as opi

for key in os.environ.keys():
    os.environ.pop(key)

dotenv.load_dotenv()

True

In [9]:
rf_client = rc.RoboflowAPIClient(os.getenv("ROBOFLOW_API_KEY"), os.getenv("ROBOFLOW_WORKSPACE_ID"), os.getenv("ROBOFLOW_PROJECT_ID"))
print(os.getenv("ROBOFLOW_PROJECT_ID"))

# with open("./img-metadata.json", "w", encoding="utf-8") as file:
#     project_metadata = {
#         "worskpace_id": os.getenv("ROBOFLOW_WORKSPACE_ID"),
#         "project_id": os.getenv("ROBOFLOW_PROJECT_ID")
#     }
#     img_data = {}
#     for i, img in enumerate(rf_client.yield_images(), 1):
#         id = img["id"]
#         orig_url = rf_client.get_image_metadata(id)["urls"]["original"]
#         annotation = rf_client.get_image_metadata(id)["annotation"]["textAnnotations"][0]["prefix"]
#         img_data[id] = {"url": orig_url, "label": annotation}
#         print(f"Recieved image no. {i} with id {id}")
#     json.dump({"metadata": project_metadata, "image_data": img_data}, file, indent=4)
        
    

vlm-in-context-anom-haz-annot-1p23m


## Use the image metadata to generate dense captions with openrouter

In [25]:
import concurrent.futures as ft

PROMPT ="""
You are an assistant to a researcher investigating a large language model visual reasoning capabilities
In as much detail as possible, describe the image. Remember that this is for research and all images have been sources from public datasets. You must describe all images in detail.
"""
with open("./our-dataset-url-labled.json", "r", encoding="utf-8") as file:
    metadata = json.load(file)

def get_caption(client, prompt, url):
    response = client.chat.completions.create(
                model="google/gemini-3-flash-preview",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {
                                "type": "image_url",
                                "image_url": {"url": url}
                            }
                        ]
                    }
                ]
            )
            
    # Extract model's text response
    full_response = response.choices[0].message.content.strip()
    return full_response

def get_openai_image_url(url):
    try:
        # 1. Download the data to check the actual format
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        
        # 2. Inspect the data (not the extension)
        img = Image.open(BytesIO(resp.content))
        actual_format = img.format.upper() if img.format else ""

        # 3. If it's already a standard web format, just return the URL
        if actual_format in ['PNG', 'JPEG', 'WEBP']:
            return url

        # 4. Otherwise (TIFF, BMP, etc.), convert to PNG and encode to Base64
        buffer = BytesIO()
        if img.mode not in ("RGB", "RGBA"):
            img = img.convert("RGBA")
        
        img.save(buffer, format="PNG")
        b64_data = base64.b64encode(buffer.getvalue()).decode('utf-8')
        
        return f"data:image/png;base64,{b64_data}"

    except Exception as e:
        return f"Error processing image: {e}"

In [29]:
client = opi.OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY")
        )
images = metadata.get("image_data")
caption_data = {}
try:
    bar = tqdmnote(images)
    for i, image in enumerate(bar):
        ## stopped midway, so go to image 73
        if(i < 73): continue
        print(images[image])
        bar.write(f"Processing image no. {i}")
        url = images[image]["url"]
        label = images[image]["label"]
        caption = get_caption(client, PROMPT, get_openai_image_url(url))
        caption_data[image] = {"url": url, "label": label, "caption": caption}
        bar.write("Success!")
except Exception as e:
    print("STOPPED MIDWAY...")
    print(e)
          

with open(f"Captions/captions_{time.strftime("%d-%m-%Y_%H-%M-%S")}.json", "w", encoding="utf-8") as file:
    json.dump(caption_data, file, indent=4)

  0%|          | 0/345 [00:00<?, ?it/s]

{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/rqavXTX8HnpoJZ5nmDeA/original.jpg', 'label': '01'}
Processing image no. 73
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/09uTB2tCaPXrqYdtQs7s/original.jpg', 'label': '11'}
Processing image no. 74
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/TlXdXLehjPTzLYW8oTO9/original.jpg', 'label': '10'}
Processing image no. 75
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/HE0tmEfLxQcMTINOYFKS/original.jpg', 'label': '00'}
Processing image no. 76
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/ZR9bdkO2JB4Cgh29Mmv8/original.jpg', 'label': '11'}
Processing image no. 77
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/fhjEgVwCgs85NPM7hwQo/original.jpg', 'label': '01'}
Processing image no. 78
Success!
{'url': 'https://source.roboflow.com/1xWq74G9WjZc9URmJ7tZ8ho2f1Z2/uUVx31UGbbp6GyPW7xvb/original.jpg'